# Qwen3-8B LoRA fine-tune for MAST Stage 1 (local, RTX 4090)

Stage C–E of `qwen-lora-finetuning-plan.md`. Assumes `train.jsonl` / `val.jsonl`
already exist (produced locally by `stage_ab_teacher_labeling.py`) in the exact
`{"messages": [{"role": system/user/assistant, "content": ...}]}` chat format,
using the **unmodified** inference-time `STAGE1_SYS`/`STAGE1_USER` template.

**Runtime:** local Jupyter, single RTX 4090 (24GB VRAM). Not Colab -- no Drive
mount, no `google.colab` calls. Run this from a Python environment with CUDA
available (check `nvidia-smi` and that `torch.cuda.is_available()` is True
before running the install cell).


## 1. Install Unsloth

Local (non-Colab) install. Unsloth's exact recommended command depends on
your installed torch/CUDA version -- check https://github.com/unslothai/unsloth#installation
if this generic form fails; the `colab-new` extra from the Colab version of
this notebook is Colab-specific and doesn't apply here.


In [ ]:
import torch
assert torch.cuda.is_available(), "no CUDA GPU visible -- check your driver/env before continuing"
print(torch.cuda.get_device_name(0), "-", round(torch.cuda.get_device_properties(0).total_memory / 2**30), "GB")

# If this fails, install the exact command from https://github.com/unslothai/unsloth#installation
# for your torch/CUDA version instead of this generic one.
%pip install --upgrade pip -q
%pip install unsloth -q
%pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes -q


## 2. Confirm Qwen3-8B support

Support tables change — the plan explicitly calls this out as something to verify
at run time, not trust from any written instructions. This just prints Unsloth's
version and attempts the 4-bit load below; if `FastLanguageModel.from_pretrained`
fails on the model name, check https://github.com/unslothai/unsloth#-model-support
and swap `MODEL_ID` for whatever the current 4-bit Qwen3-8B tag is.

In [ ]:
import unsloth
print("unsloth version:", unsloth.__version__)


## 3. Point at your local train.jsonl / val.jsonl

Same machine, no upload/Drive needed -- just point `DATA_DIR` at wherever
`train.jsonl`/`val.jsonl` already are (defaults to the current working
directory, matching `mast_window_ablation.ipynb`'s convention -- open this
notebook from the `Experiment4` folder, or edit the path below).


In [ ]:
from pathlib import Path

DATA_DIR = Path.cwd()   # <-- edit if you're not running this from the Experiment4 folder
TRAIN_PATH = DATA_DIR / 'train.jsonl'
VAL_PATH = DATA_DIR / 'val.jsonl'
OUT_DIR = DATA_DIR / 'runs' / 'qwen3_8b_lora_v1'
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_PATH.exists(), f"missing {TRAIN_PATH}"
assert VAL_PATH.exists(), f"missing {VAL_PATH}"
print(f"train: {TRAIN_PATH}\nval:   {VAL_PATH}\nout:   {OUT_DIR}")


## 4. Load base model in 4-bit + attach LoRA

`max_seq_length=6144` -- measured against the actual `train.jsonl`/`val.jsonl`
with the real Qwen3 tokenizer: max example is 5,137 tokens, p99 is 3,582. The
original 4096 cutoff was silently truncating the 2 longest examples.

This is sized for a single RTX 4090 (**24GB VRAM**, not the 70GB system RAM
you also have -- that's separate from GPU VRAM and doesn't change what fits
on the card). `r=32` is fine (LoRA params are a tiny fraction of memory
regardless of rank). `use_gradient_checkpointing=True` (standard recompute,
not the `"unsloth"` CPU-offload variant) trades a bit of speed for VRAM
headroom -- on 24GB with seq_length this long, keep it on; a 4090's raw
compute is strong enough that the recompute cost is cheap relative to the
VRAM it buys back.

Unsloth's 4-bit dynamic quant loads directly from the HF hub; no separate
quantization step needed here (that only matters for the MLX conversion
later, in Stage E).


In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_ID = "unsloth/Qwen3-8B-bnb-4bit"   # verify current tag in Unsloth's model list before running
MAX_SEQ_LENGTH = 6144   # covers the measured max (5,137 tok) with margin -- see markdown above

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,                # keep alpha = 2*r ratio
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing=True,   # standard recompute -- see markdown above
    random_state=3407,
)


## 5. Load + format the dataset

Applies Qwen3's own chat template via the tokenizer (`enable_thinking=False`,
matching every inference call in the prompted pipeline) so training sees
*exactly* the same token sequence shape as eval will. Loss is masked to the
assistant turn only (`train_on_responses_only`) — otherwise the model would
also be trained to predict the system/user prompt tokens, which wastes
capacity and dilutes the signal on the few hundred assistant tokens that
actually matter.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, train_on_responses_only

tokenizer = get_chat_template(tokenizer, chat_template="qwen3")

def formatting_func(examples):
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False, enable_thinking=False)
        for convo in examples["messages"]
    ]
    return {"text": texts}

raw_ds = load_dataset("json", data_files={"train": str(TRAIN_PATH), "val": str(VAL_PATH)})
train_ds = raw_ds["train"].map(formatting_func, batched=True)
val_ds = raw_ds["val"].map(formatting_func, batched=True)
print(train_ds)
print(val_ds)
print("\n--- sample formatted example ---\n")
print(train_ds[0]["text"][:2000])


## 6. Train

Config lifted from Stage D of the plan, sized for a single 24GB RTX 4090:
`per_device_train_batch_size=2` with `gradient_accumulation_steps=8` (effective
batch 16, same as the plan's original tuning). At `max_seq_length=6144` this
is already a meaningfully bigger workload per step than the plan's original
4096 assumption, so this stays conservative rather than pushing batch size up
-- if `nvidia-smi` shows comfortable headroom during a run, batch size 4 /
accum 4 is worth trying for speed, same effective batch either way.

With only ~1,900 step-examples, **overfitting is the risk, not underfitting**
-- `eval_strategy="steps"` + `eval_steps=50` so you can watch val loss diverge
from train loss and stop early if needed, rather than committing to the full
3 epochs blind.


In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=str(OUT_DIR / "checkpoints"),
    per_device_train_batch_size=2,     # conservative for 24GB at max_seq_length=6144 -- see markdown above
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,     # effective batch 16
    learning_rate=1e-4,                # try 2e-4 if underfitting after this run
    num_train_epochs=3,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_seq_length=MAX_SEQ_LENGTH,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    dataset_text_field="text",
    packing=False,   # keep each step-example as its own sequence; packing would blur step boundaries
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
)

# Mask the loss to assistant tokens only -- see markdown above.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)


In [ ]:
trainer_stats = trainer.train()
print(trainer_stats)


## 6b. Sanity-check training curves

If eval_loss starts climbing while train_loss keeps falling, that's the
overfitting the plan warns about — stop there rather than letting all 3
epochs run; `load_best_model_at_end=True` above already protects you from
shipping the worst checkpoint, but look at this plot to decide whether even
fewer epochs would have been better for a re-run.

In [ ]:
import matplotlib.pyplot as plt

log = trainer.state.log_history
train_pts = [(e["step"], e["loss"]) for e in log if "loss" in e and "eval_loss" not in e]
eval_pts = [(e["step"], e["eval_loss"]) for e in log if "eval_loss" in e]

plt.figure(figsize=(7, 4))
if train_pts:
    xs, ys = zip(*train_pts)
    plt.plot(xs, ys, label="train_loss")
if eval_pts:
    xs, ys = zip(*eval_pts)
    plt.plot(xs, ys, label="eval_loss", marker="o")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("Qwen3-8B LoRA fine-tune")
plt.savefig(OUT_DIR / "loss_curve.png", dpi=120)
plt.show()


## 7. Quick qualitative check before merging

Generate on a handful of held-out **val** examples and eyeball that output is
still well-formed `REASONING:`/`MODE:`/`EVIDENCE:` text (plan Stage E4 -- LoRA
fine-tuning at too high an LR can destabilize output formatting).
`max_new_tokens=450` matches `MAX_NEW_TOKENS["stage1"]` in
`mast_window_ablation.ipynb`, bumped there for the same reason (the
`REASONING:` field adds ~100-150 tokens before `MODE:`/`EVIDENCE:`).


In [ ]:
FastLanguageModel.for_inference(model)

import random
random.seed(0)
sample_idxs = random.sample(range(len(val_ds)), min(5, len(val_ds)))

for i in sample_idxs:
    convo = raw_ds["val"][i]["messages"]
    prompt_msgs = convo[:2]  # system + user, drop the gold assistant turn
    inputs = tokenizer.apply_chat_template(
        prompt_msgs, tokenize=True, add_generation_prompt=True, enable_thinking=False,
        return_tensors="pt",
    ).to(model.device)
    out = model.generate(input_ids=inputs, max_new_tokens=450, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
    gen_text = tokenizer.decode(out[0, inputs.shape[1]:], skip_special_tokens=True)
    print(f"=== val example {i} ===")
    print("GOLD:", convo[2]["content"][:300])
    print("PRED:", gen_text.strip()[:300])
    print()


## 8. Save adapter, merge, and export a merged HF checkpoint

This is Stage E1. You're training on CUDA/HF here but your inference pipeline
runs on MLX on Apple Silicon -- the merged HF checkpoint below is the artifact
you transfer to your Mac (scp/rsync/USB -- whatever's convenient between your
two machines) and convert in Stage E3 (not the raw LoRA adapter).


In [ ]:
ADAPTER_DIR = OUT_DIR / "lora_adapter"
MERGED_DIR = OUT_DIR / "merged_fp16"

model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print(f"LoRA adapter saved to {ADAPTER_DIR}")

# Use Unsloth's own merge helper, NOT the raw PEFT model.merge_and_unload(). Unsloth patches the
# model with custom fused kernels/layers, and calling merge_and_unload() directly on a
# FastLanguageModel is a known source of silently incorrect merges. save_pretrained_merged handles
# the Unsloth-specific state correctly in one call.
model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method="merged_16bit")
print(f"Merged HF checkpoint saved to {MERGED_DIR}")


## 9. Next step (run on your Mac, not here)

`MERGED_DIR` is already a normal local directory on this machine -- no zip or
download step needed. Copy it to your Mac however you'd copy any folder
between two machines on your network (`rsync`, `scp`, a shared drive, etc.),
then continue with Stage E3–E4 below.


In [ ]:
import subprocess
print(f"merged checkpoint ready at: {MERGED_DIR}")
print(subprocess.run(["du", "-sh", str(MERGED_DIR)], capture_output=True, text=True).stdout.strip())
print("\nCopy this directory to your Mac, e.g.:")
print(f"  rsync -avz {MERGED_DIR}/ your-mac-user@your-mac-hostname:/path/to/merged_fp16/")


## 10. Next step (run on your Mac, not here)

Stage E3–E4 from the plan, to run locally with `mlx-lm` once `merged_fp16/`
is on your Mac:

```bash
pip install -U mlx-lm

mlx_lm.convert \
    --hf-path /path/to/merged_fp16 \
    --mlx-path /path/to/qwen3-8b-mast-lora-mlx \
    -q --q-bits 4

# sanity check a few steps before the full eval -- confirm MODE:/EVIDENCE: output
# is still well-formed (point mast_window_ablation.ipynb's MLX_MODEL_ID at the
# new --mlx-path and re-run a handful of trajectories)
```

Then re-point `MLX_MODEL_ID` in `mast_window_ablation.ipynb` at the new
`qwen3-8b-mast-lora-mlx` directory and re-run the window=0 / window=3 reports
on the untouched 56-trajectory eval set (plan Stage F).
